# CIFAR-10: **CPU vs GPU**, head to head

Runs the identical resnet-18 + engine on both devices for a few epochs on a subset (a *throughput* demo, not a full train) and contrasts them. The only input that changes is `device` — everything else comes from `../common`.

Expect: the GPU an order of magnitude faster in img/s; the CPU held back not by under-utilization but by being a CPU (its knobs still buy ~2.3x, see [`cifar10_cpu_train.ipynb`](cifar10_cpu_train.ipynb)).

In [ ]:
# -- Shared setup: ../common has the pipeline + engine, ../a1-imagenet32 has models.py --
import os, sys
for rel in ('../common', '../a1-imagenet32'):
    p = os.path.normpath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from gpu_check import set_seed
from cifar_pipeline import load_cifar10_arrays, make_loaders
from train_engine import train
import models as M
set_seed(42)

In [ ]:
import time, torch
trx, tryy, tex, tey = load_cifar10_arrays()
# subset so the CPU side finishes quickly — this is a throughput comparison, not full training
trx, tryy, tex, tey = trx[:10000], tryy[:10000], tex[:5000], tey[:5000]
EPOCHS = 3
results = {}
for device in ['cpu'] + (['cuda'] if torch.cuda.is_available() else []):
    ti, vi, cfg = make_loaders(device, trx, tryy, tex, tey)
    m = M.build('resnet18', num_classes=10)
    t0 = time.time()
    hist, best = train(m, ti, vi, device, epochs=EPOCHS,
                       channels_last=cfg['channels_last'], amp_dtype=cfg['amp_dtype'],
                       lr=0.1 * cfg['batch_size'] / 256, log=lambda *_: None)
    steady = sum(hist['img_s'][1:]) / max(1, len(hist['img_s']) - 1)   # drop epoch 1
    results[device] = dict(img_s=steady, wall=time.time() - t0, cfg=cfg)
    print(f'{device:4s} {cfg["backend"]:14s} {steady:8,.0f} img/s')

## The difference

In [ ]:
gpu = results.get('cuda'); cpu = results.get('cpu')
print(f'{"device":6s} {"backend":15s} {"img/s":>10s} {"rel":>8s}')
base = cpu['img_s'] if cpu else 1
for d, r in results.items():
    print(f'{d:6s} {r["cfg"]["backend"]:15s} {r["img_s"]:>10,.0f} {r["img_s"]/base:>7.1f}x')
if gpu and cpu:
    print(f'\nGPU is {gpu["img_s"]/cpu["img_s"]:.0f}x the CPU throughput on this workload.')

In [ ]:
import matplotlib.pyplot as plt
labels = [f'{d}\n{results[d]["cfg"]["backend"]}' for d in results]
vals = [results[d]['img_s'] for d in results]
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, vals, color=['#c44', '#4a4'][:len(vals)])
ax.set_ylabel('images / sec (steady state)'); ax.set_title('CIFAR-10 resnet18: CPU vs GPU')
ax.bar_label(bars, fmt='%.0f')
plt.tight_layout(); plt.show()